## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load the data

In [ ]:
train = pd.read_csv("train.csv")

print("Shape:", train.shape)
train.head()

## 3. Explore the data

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isnull().sum()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(train["accident_risk"], bins=40, color="steelblue", edgecolor="white")
plt.title("Distribution of accident risk")
plt.xlabel("accident_risk")
plt.ylabel("count")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["road_type", "weather", "lighting"]):
    train.groupby(col)["accident_risk"].mean().plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Average risk by {col}")
    ax.set_ylabel("mean accident_risk")

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ["num_lanes", "curvature", "speed_limit", "num_reported_accidents", "accident_risk"]
train[numeric_cols].corr()["accident_risk"].sort_values(ascending=False)

In [ ]:
X = train.drop(columns=["id", "accident_risk"])
y = train["accident_risk"]

text_cols = ["road_type", "lighting", "weather", "time_of_day"]
X = pd.get_dummies(X, columns=text_cols, dtype=int)

bool_cols = ["road_signs_present", "public_road", "holiday", "school_season"]
X[bool_cols] = X[bool_cols].astype(int)

X.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:  ", len(X_train))
print("Validation rows:", len(X_val))

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

lin_pred = lin_model.predict(X_val)

lin_rmse = np.sqrt(mean_squared_error(y_val, lin_pred))
lin_r2 = r2_score(y_val, lin_pred)

print(f"Linear Regression -> RMSE: {lin_rmse:.4f} | R2: {lin_r2:.4f}")

In [ ]:
X_small = X_train.sample(n=min(100_000, len(X_train)), random_state=42)
y_small = y_train.loc[X_small.index]

rf_model = RandomForestRegressor(
    n_estimators=100, 
    max_depth=12,      
    n_jobs=-1,          
    random_state=42
)
rf_model.fit(X_small, y_small)

rf_pred = rf_model.predict(X_val)

rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_r2 = r2_score(y_val, rf_pred)

print(f"Random Forest -> RMSE: {rf_rmse:.4f} | R2: {rf_r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [lin_rmse, rf_rmse],
    "R2": [lin_r2, rf_r2],
})
results

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_val[:2000], rf_pred[:2000], alpha=0.3, s=10)
plt.plot([0, 1], [0, 1], color="red")
plt.xlabel("Actual risk")
plt.ylabel("Predicted risk")
plt.title("Random Forest: predicted vs actual")
plt.show()

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 6))
importances.plot(kind="barh", color="steelblue")
plt.title("Feature importance (Random Forest)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
test = pd.read_csv("test.csv")
X_test = test.drop(columns=["id"])
X_test = pd.get_dummies(X_test, columns=text_cols, dtype=int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

X_test = X_test.reindex(columns=X.columns, fill_value=0)

test_pred = rf_model.predict(X_test)

submission = pd.DataFrame({"id": test["id"], "accident_risk": test_pred})
submission.to_csv("submission.csv", index=False)
submission.head()